In [33]:
import pandas as pd
import numpy as np

individus = pd.read_csv("donnees/individus.csv")
groupes = pd.read_csv("donnees/groupes.csv")
aliments = pd.read_csv("donnees/aliments.csv")
constituants = pd.read_csv("donnees/constituants.csv")
composition = pd.read_csv("donnees/composition.csv")
consommations = pd.read_csv("donnees/consommations.csv")

Tables = [individus, groupes, aliments, constituants, composition, consommations]

In [25]:
for df in [individus, groupes, aliments, constituants, composition, consommations]:
    print(df.head())
    #print(df.info())
    print('\n')

   id_individu sexe  age  poids_kg  taille_cm
0            1    F   22      58.5        165
1            2    M   35      78.0        178
2            3    F   47      68.0        162
3            4    M   61      82.5        175
4            5    F   29      62.0        170


  code_groupe                          libelle
0         G01  Céréales et produits céréaliers
1         G02                           Fruits
2         G03                          Légumes
3         G04                Produits laitiers
4         G05              Viandes et poissons


   code_aliment           libelle code_groupe
0          1001        Pain blanc         G01
1          1002      Pâtes cuites         G01
2          2001             Pomme         G02
3          3001     Carotte cuite         G03
4          4001  Lait demi-écrémé         G04


   code_const        nom      unite
0           1    Energie  kcal/100g
1           2  Proteines     g/100g
2           3    Lipides     g/100g
3           4   

1. Quelle table contient les individus ?  
La table individus et consomation contient  `id_ind`  

2. Quelle table décrit les aliments ?  
Les tables constituants et composition  

3. Quelle table donne les valeurs nutritionnelles ?  
La table composition donne la teneur de chaque constituant pour chaque aliment  

4. Pourquoi `composition` possède-t-elle deux codes ?  
Elle a les codes aliments et les codes constituants

In [22]:
def rechercher_aliment(texte):
    """Retourne les aliments dont le libellé contient texte."""
    return aliments[aliments['libelle'].str.contains(texte, case=False)]


rechercher_aliment("pain")

,code_aliment,libelle,code_groupe
0,1001,Pain blanc,G01


In [ ]:
def composition_aliment(code_aliment):
    """Retourne la composition d'un aliment, avec nom et unité du constituant."""
    resultat = composition[composition["code_aliment"] == code_aliment].merge(
        constituants,
        on="code_const",
        how="left"
    )

    return resultat

composition_aliment(1001)

,code_aliment,code_const,teneur,nom,unite
0,1001,1,265.0,Energie,kcal/100g
1,1001,2,8.5,Proteines,g/100g
2,1001,3,1.2,Lipides,g/100g
3,1001,4,49.0,Glucides,g/100g
4,1001,5,2.7,Fibres,g/100g


In [78]:
def apport_aliment(code_aliment, quantite_g, precis=False):
    """Calcule les apports pour une quantité donnée en grammes."""
    teneur = composition[composition["code_aliment"] == code_aliment].assign(
        code_const=lambda df: df["code_const"].map(
            constituants.set_index("code_const")["nom"]
        )
    )[["code_const", "teneur"]]

    teneur["teneur"] = teneur["teneur"] * quantite_g / 100

    apport = teneur.rename(
    columns={"code_const": "Nom", "teneur": "apport (g)"}
)
    if precis == False:
        return apport
    
    return apport[apport["Nom"] == precis]

apport_aliment(1001, 60)

,Nom,apport (g)
0,Energie,159.00
1,Proteines,5.10
2,Lipides,0.72
3,Glucides,29.40
4,Fibres,1.62


In [94]:
def apport_aliment(code_aliment, quantite_g, precis=False):
    a = composition[composition["code_aliment"] == code_aliment].merge(
        constituants, on="code_const", how="left")
    apport = a[["nom","teneur"]]
    apport["teneur"] = apport["teneur"] * quantite_g / 100

    if precis == False:
            return apport
    
    return apport[apport["nom"] == precis]

apport_aliment(1001, 60, "Proteines")

,nom,teneur
1,Proteines,5.1


### Partie D.
1. sélectionner les consommations de l'individu ;
2. retrouver les aliments ;
3. retrouver leur composition ;
4. calculer l'apport correspondant à chaque consommation ;
5. additionner les apports pour chaque constituant.

In [107]:
def apports_individu(id_individu):
    """Calcule les apports nutritionnels totaux d'un individu.
    """
    consommations_individu = consommations[consommations["id_individu"] == id_individu][["code_aliment","quantite_g"]] #recuperer les consommations de l'individu

    apport = apport_aliment(consommations_individu.iloc[0]["code_aliment"], consommations_individu.iloc[0]["quantite_g"]) #initialiser l'apport avec le premier aliment consommé
    apport["teneur"] = 0 #initialiser la colonne teneur à 0

    for code, quantit in zip(consommations_individu["code_aliment"], consommations_individu["quantite_g"]): #additionner les apports de chaque aliment consommé
        apport_temp = apport_aliment(code, quantit)
        apport["teneur"] += apport_temp["teneur"]

    return apport

apports_individu(1)

,nom,teneur
0,Energie,864.00
1,Proteines,56.55
2,Lipides,15.62
3,Glucides,118.50
4,Fibres,11.46


In [110]:
def apports_individu(id_individu):
    """Calcule les apports nutritionnels totaux d'un individu."""
    consommations_individu = consommations[consommations["id_individu"] == id_individu][["code_aliment", "quantite_g"]]

    if consommations_individu.empty:
        return pd.DataFrame(columns=["nom", "teneur"])

    apport = (consommations_individu.merge(
        composition, on="code_aliment", how="left"
        ).merge(constituants, on="code_const", how="left") )

    apport["apport"] = apport["teneur"] * apport["quantite_g"] / 100

    return (
        apport.groupby("nom", as_index=False)["apport"]
        .sum()
        .rename(columns={"apport": "teneur"})
    )


apports_individu(1)

,nom,teneur
0,Energie,864.00
1,Fibres,11.46
2,Glucides,118.50
3,Lipides,15.62
4,Proteines,56.55


### Partie E.
On retrouve `apports_individu` dans la ligne "total journalier" de `apports_par_repas`

In [118]:
def apports_par_repas(id_individu):
    """Calcule les apports nutritionnels par repas et ajoute le total journalier."""
    consommations_individu = consommations[
        consommations["id_individu"] == id_individu
    ][["code_aliment", "quantite_g", "repas"]]

    if consommations_individu.empty:
        return pd.DataFrame(columns=["repas"])

    apport = (
        consommations_individu
        .merge(composition, on="code_aliment", how="left")
        .merge(constituants, on="code_const", how="left")
    )

    apport["apport"] = apport["teneur"] * apport["quantite_g"] / 100

    resultat = (
        apport.pivot_table(
            index="repas",
            columns="nom",
            values="apport",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )

    total = resultat.drop(columns="repas").sum()
    total["repas"] = "total journalier"

    return pd.concat(
        [resultat, pd.DataFrame([total])],
        ignore_index=True
    )


apports_par_repas(1)


,repas,Energie,Fibres,Glucides,Lipides,Proteines
0,collation,78.0,3.60,18.0,0.30,0.45
1,dejeuner,533.0,6.24,61.5,11.60,44.60
2,petit_dejeuner,253.0,1.62,39.0,3.72,11.50
3,total journalier,864.0,11.46,118.5,15.62,56.55


### Partie F.
Avec les fonctions :
- `rechercher_aliment(texte)`
- `composition_aliment(code_aliment)`
- `apport_aliment(code_aliment, quantite_g, precis=False)`
- `apports_individu(id_individu)`
- `apports_par_repas(id_individu)`

1. Combien d'aliments différents sont consommés par l'individu 1 ?  
On ne peut pas savoir avec les fonctions que l'on a.  

2. Quel est son apport total en énergie ?  
864 kcal  

3. Quel est son apport total en protéines ?
4. Quel repas lui apporte le plus d'énergie ?
5. Quel aliment contribue le plus à son apport protéique ?
6. Quel individu consomme la plus grande quantité totale d'aliments ?
7. Quel individu possède l'apport énergétique total le plus élevé ?

In [119]:
# 2.
print(apports_par_repas(1))

              repas  Energie  Fibres  Glucides  Lipides  Proteines
0         collation     78.0    3.60      18.0     0.30       0.45
1          dejeuner    533.0    6.24      61.5    11.60      44.60
2    petit_dejeuner    253.0    1.62      39.0     3.72      11.50
3  total journalier    864.0   11.46     118.5    15.62      56.55
